In [1]:
import torch.nn as nn
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import string
import numpy as np
import pandas as pd
import seaborn as sns
from torch.utils.tensorboard import SummaryWriter
import re
import tiktoken
from torch.utils.data import TensorDataset, DataLoader
from pprint import pprint
from tqdm import tqdm
import time
from pathlib import Path

In [ ]:
torch.random.manual_seed(1234)

enc = tiktoken.get_encoding('gpt2')
vocab_size = enc.n_vocab
print(vocab_size)

context_length = 128
embedding_dim = 512
batch_size=32

model_dir = "./models"

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(device)


def save_model(model, file_name="model.torch"):
    with open(Path(model_dir, file_name), 'wb') as f:
        torch.save(model, f)

def load_model(file_name="model.torch"):
    with open(Path(model_dir, file_name), 'rb') as f:
        model = torch.load(f, weights_only=False)
    return model

def tokenise_text(text):
    tokens = torch.tensor(enc.encode(text), device=device)
    return tokens

def load_data():

    with open('complete-jane-austen.txt') as f:
        content = f.read()
        content = ' '.join(content.split())
        print(len(content))
        print(content[:100])
        encoded_text = tokenise_text(content)
        print(encoded_text[:100])
        print(len(encoded_text))

    xs = []
    ys = []
    for i in range(0, len(encoded_text)-context_length):
        x_chunk = encoded_text[i:i+context_length]
        y_chunk = encoded_text[i+1:i+context_length+1]

        xs.append(x_chunk)
        ys.append(y_chunk)

    X = torch.stack(xs)
    Y = torch.stack(ys)
    split_index = int(X.shape[0]*0.99)
    X_train = X[:split_index]
    Y_train = Y[:split_index]
    X_val = X[split_index:]
    Y_val = Y[split_index:]
    
    dataset_train = TensorDataset(X_train, Y_train)
    dataset_val = TensorDataset(X_val, Y_val)

    loader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
    loader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=True)
    
    return loader_train, loader_val



def generate(model, max_words, prompt_text):
    
    idx = tokenise_text(prompt_text).unsqueeze(0)
    if idx.shape[0]< context_length:
        prompt_text+=(' '*(context_length-idx.shape[0]))
        idx = tokenise_text(prompt_text).unsqueeze(0)
    
    for _ in range(max_words):
        idx_cond = idx[:, -context_length:]
        logits = model(idx_cond)
        logits = logits[:, -1, :]

        probs = torch.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, next_idx), dim=1)

        output = enc.decode(idx[0].tolist()) 
    return output

# generate(model, 20, ' ')

50257
cuda


In [4]:
class Head(nn.Module):
    def __init__(self, head_dimension, n_embd):
        super().__init__()

        self.query = nn.Linear(n_embd, head_dimension, bias=False)
        self.key = nn.Linear(n_embd, head_dimension, bias=False)
        self.value = nn.Linear(n_embd, head_dimension, bias=False)
        
        self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))
        

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        wei = q @ k.transpose(-2,-1)*(C**-0.5)
        wei = wei.masked_fill(self.tril[:T, :T]==0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        out =  wei @ v

        return out
    
    
class TransformerNameGenerator(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd):
        super().__init__()
        self.n_embd = n_embd
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(context_length, n_embd)
        self.head_1 = Head(self.n_embd, self.n_embd)
        self.head_2 = Head(self.n_embd, self.n_embd)
        self.llm_model = nn.Linear(n_embd, vocab_size)
        self.ln_1 = nn.LayerNorm(n_embd)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.ln_3 = nn.LayerNorm(n_embd)

        self.ffn = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.ReLU(), nn.Linear(4*n_embd, n_embd))
            

    def forward(self, idx):
        B, T = idx.shape
 
        token_embedding = self.token_embedding(idx) # B, T, embd_n
 
        pos = torch.arange(T, device=device)
 
        positional_embeddings = self.position_embedding(pos) # T, nemb
 
        x = token_embedding + positional_embeddings
        
        x = x + self.head_1(self.ln_1(x))

        x = x + self.head_2(self.ln_2(x))

        x = x + self.ffn(self.ln_3(x))
        
        logits = self.llm_model(x)

        return logits


In [5]:
model = TransformerNameGenerator(vocab_size, context_length, embedding_dim)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.01)

In [ ]:
loader_train, loader_val = load_data()

In [ ]:
writer = SummaryWriter()

pbar = tqdm(loader_train)
epochs = 1

model.train()
tokens_per_second = 0
for epoch in range(epochs):
    for i, (xb, yb) in enumerate(pbar):
        torch.cuda.synchronize()
        start = time.time()
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model.forward(xb)
        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = yb.view(B*T)
        loss = F.cross_entropy(logits, targets)
        writer.add_scalar("Loss/train", loss, i)
        if not i%100:
            # print(loss.item(), f'Epoch: {epoch}, Batch: {i}')
            with torch.no_grad():
                model.eval()
                val_loss=0
                for xb, yb in loader_val:
                    if xb.shape[0]==batch_size:
                        val_loss += F.cross_entropy(model.forward(xb).view(B*T, C), yb.view(B*T)) 
                pbar.set_postfix(val_loss=(val_loss/len(loader_val)).item(), loss=loss.item(), tokens_per_second=tokens_per_second)
                writer.add_scalar("Loss/val", val_loss/len(loader_val), i)

                output = generate(model, 50, "")
                print(output)

                model.train()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        torch.cuda.synchronize()
        step_time = time.time() - start
        tokens_per_second = batch_size*context_length/step_time
        # print(tokens_per_second)

        

  0%|          | 0/30069 [00:22<?, ?it/s, loss=11.2, tokens_per_second=0, val_loss=11.1]

                                                                                                                               rav jewelryprotected BombCook Printing Pennyinanceagonist Moroccan discipline 2050 dictator attaching chillyductaml Ing)-- Gloves revolver sterling greedy busheva dominating Unicode KearJC Boomirds Jerome colonies 1999dimension fillsaco Used overridebad EntBrave inspectorssteel obeSky Pebble Hobby skewed bra


  0%|          | 100/30069 [05:16<21:29:19,  2.58s/it, loss=5.73, tokens_per_second=1.57e+3, val_loss=5.8]

                                                                                                                                struck too longer- Thursday inclination, set of,) so so a variation-- up twice to," said kept is living subs were ill of her to the from the evening, and nobody sees a very Hus, and the first. I often. When concern


  1%|          | 200/30069 [10:10<19:43:50,  2.38s/it, loss=5.25, tokens_per_second=2.25e+3, val_loss=5.38]

                                                                                                                                is connection except. She by relent promise of other pride'?" "your to see with a ball, which the familyr render the precise a clothes ofGeneral interrupted away applied, by Mr in what long in my notice that he adviceation to the


  1%|          | 300/30069 [15:03<19:38:00,  2.37s/it, loss=5.13, tokens_per_second=1.7e+3, val_loss=5.19]  

                                                                                                                                may be a little dared arguedShe pen in the only object. Emma, he might) trouble her companion dear "From vexation, but if I ugly, come receivingNo so an hour on the evening anOnnet within dull _ feature indeed came


  1%|          | 339/30069 [16:41<21:01:27,  2.55s/it, loss=5.13, tokens_per_second=1.7e+3, val_loss=5.19] 

In [ ]:
save_model(model, file_name="multihead.torch")